## 1. Setup and Configuration

In [17]:
import pandas as pd
import numpy as np
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
import time

# Add backend to path
backend_path = Path('../backend').resolve()
sys.path.insert(0, str(backend_path))

# Load environment variables
load_dotenv(backend_path / '.env')

print(f"Backend path: {backend_path}")
print(f"OpenAI API Key loaded: {'Yes' if os.getenv('OPENAI_API_KEY') else 'No'}")
print(f"Pinecone API Key loaded: {'Yes' if os.getenv('PINECONE_API_KEY') else 'No'}")

Backend path: /home/chris/coding-task/productlens-ai/backend
OpenAI API Key loaded: Yes
Pinecone API Key loaded: Yes


## 2. Load Cleaned Products

In [18]:
# Load cleaned products
cleaned_path = backend_path / 'data' / 'cleaned' / 'products_cleaned.csv'
df_products = pd.read_csv(cleaned_path)

print(f"Loaded {len(df_products)} products")
print(f"Columns: {df_products.columns.tolist()}")
df_products.head()

Loaded 4145 products
Columns: ['StockCode', 'Description', 'UnitPrice', 'PriceMin', 'PriceMax', 'PriceStd', 'TotalQuantitySold', 'PrimaryCountry', 'TransactionCount']


,StockCode,Description,UnitPrice,PriceMin,PriceMax,PriceStd,TotalQuantitySold,PrimaryCountry,TransactionCount
0,10002,INFLATABLE POLITICAL GLOBE,1.09,0.85,1.66,0.367792,860,United Kingdom,71
1,10080,GROOVY CACTUS INFLATABLE,0.41,0.39,0.85,0.098072,303,United Kingdom,22
2,10120,DOGGY RUBBER,0.21,0.21,0.21,0.000000,192,United Kingdom,29
3,10123C,HEARTS WRAPPING TAPE,0.65,0.65,0.65,0.000000,5,United Kingdom,3
4,10124A,SPOTS ON RED BOOKCOVER TAPE,0.42,0.42,0.42,0.000000,16,United Kingdom,5


In [19]:
# Create text representation for embedding
# Combine description with price context for richer embeddings
def create_embedding_text(row):
    price_category = 'budget' if row['UnitPrice'] < 5 else 'mid-range' if row['UnitPrice'] < 20 else 'premium'
    return f"{row['Description']}. A {price_category} priced product."

df_products['embedding_text'] = df_products.apply(create_embedding_text, axis=1)

print("Sample embedding texts:")
for text in df_products['embedding_text'].head(5):
    print(f"  - {text}")

Sample embedding texts:
  - INFLATABLE POLITICAL GLOBE. A budget priced product.
  - GROOVY CACTUS INFLATABLE. A budget priced product.
  - DOGGY RUBBER. A budget priced product.
  - HEARTS WRAPPING TAPE. A budget priced product.
  - SPOTS ON RED BOOKCOVER TAPE. A budget priced product.


## 3. Initialize OpenAI and Pinecone

In [20]:
import os
import requests
from pinecone import Pinecone

# Initialize Pinecone
pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))

# Get the index
index_name = os.getenv('PINECONE_INDEX_NAME', 'product-recommendations')
index = pc.Index(index_name)

# Check index stats
stats = index.describe_index_stats()
print(f"Index: {index_name}")
print(f"Current vector count: {stats.total_vector_count}")
print(f"Dimension: {stats.dimension}")

Index: product-recommendations
Current vector count: 4000
Dimension: 1536


## 4. Generate Embeddings

In [21]:
def get_embeddings_batch(texts, model="text-embedding-3-small"):
    """Generate embeddings for a batch of texts using requests."""
    response = requests.post(
        "https://api.openai.com/v1/embeddings",
        headers={
            "Authorization": f"Bearer {os.getenv('OPENAI_API_KEY')}",
            "Content-Type": "application/json"
        },
        json={"input": texts, "model": model}
    )
    response.raise_for_status()
    return [item["embedding"] for item in response.json()["data"]]

# Test with a single embedding first
test_text = df_products['embedding_text'].iloc[0]
test_embedding = get_embeddings_batch([test_text])[0]

print(f"Test text: {test_text}")
print(f"Embedding dimension: {len(test_embedding)}")
print(f"First 5 values: {test_embedding[:5]}")

Test text: INFLATABLE POLITICAL GLOBE. A budget priced product.
Embedding dimension: 1536
First 5 values: [-0.004441667, 0.032389317, -0.01520103, 0.042622205, -0.0065290276]


In [22]:
# Generate embeddings in batches
BATCH_SIZE = 100
all_embeddings = []

texts = df_products['embedding_text'].tolist()
total_batches = (len(texts) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"Generating embeddings for {len(texts)} products in {total_batches} batches...")

for i in range(0, len(texts), BATCH_SIZE):
    batch_texts = texts[i:i + BATCH_SIZE]
    batch_num = i // BATCH_SIZE + 1
    
    try:
        batch_embeddings = get_embeddings_batch(batch_texts)
        all_embeddings.extend(batch_embeddings)
        
        if batch_num % 10 == 0 or batch_num == total_batches:
            print(f"  Batch {batch_num}/{total_batches} completed ({len(all_embeddings)} embeddings)")
        
        # Small delay to avoid rate limits
        time.sleep(0.1)
        
    except Exception as e:
        print(f"  Error in batch {batch_num}: {e}")
        time.sleep(5)  # Wait longer on error
        batch_embeddings = get_embeddings_batch(batch_texts)
        all_embeddings.extend(batch_embeddings)

print(f"\nGenerated {len(all_embeddings)} embeddings")

Generating embeddings for 4145 products in 42 batches...
  Batch 10/42 completed (1000 embeddings)
  Batch 20/42 completed (2000 embeddings)
  Batch 30/42 completed (3000 embeddings)
  Batch 40/42 completed (4000 embeddings)
  Batch 42/42 completed (4145 embeddings)

Generated 4145 embeddings


## 5. Upload to Pinecone

In [24]:
import re

def sanitize_id(text):
    """Remove non-ASCII characters and create a valid Pinecone ID."""
    # Remove non-ASCII characters
    ascii_only = text.encode('ascii', 'ignore').decode('ascii')
    # Replace any remaining special chars with underscore
    cleaned = re.sub(r'[^a-zA-Z0-9_-]', '_', ascii_only)
    return cleaned

# Prepare vectors for upsert
vectors_to_upsert = []

for idx, row in df_products.iterrows():
    # Sanitize stock code for vector ID
    clean_stock = sanitize_id(str(row['StockCode']))
    vector_id = f"product_{clean_stock}_{idx}"
    embedding = all_embeddings[idx]
    
    # Get quantity from TotalQuantitySold column
    quantity = int(row.get('TotalQuantitySold', 0)) if pd.notna(row.get('TotalQuantitySold')) else 0
    
    # Get country from PrimaryCountry column
    country = str(row.get('PrimaryCountry', 'Unknown'))
    
    metadata = {
        'stock_code': str(row['StockCode']),
        'description': row['Description'],
        'unit_price': float(row['UnitPrice']),
        'country': country,
        'quantity': quantity,
        'transaction_count': int(row.get('TransactionCount', 0))
    }
    
    vectors_to_upsert.append({
        'id': vector_id,
        'values': embedding,
        'metadata': metadata
    })

print(f"Prepared {len(vectors_to_upsert)} vectors for upsert")
print(f"Sample IDs: {[v['id'] for v in vectors_to_upsert[:3]]}")
print(f"Sample metadata: {vectors_to_upsert[0]['metadata']}")

Prepared 4145 vectors for upsert
Sample IDs: ['product_10002_0', 'product_10080_1', 'product_10120_2']
Sample metadata: {'stock_code': '10002', 'description': 'INFLATABLE POLITICAL GLOBE', 'unit_price': 1.09, 'country': 'United Kingdom', 'quantity': 860, 'transaction_count': 71}


In [25]:
# Upsert in batches
UPSERT_BATCH_SIZE = 100
total_upsert_batches = (len(vectors_to_upsert) + UPSERT_BATCH_SIZE - 1) // UPSERT_BATCH_SIZE

print(f"Uploading {len(vectors_to_upsert)} vectors to Pinecone in {total_upsert_batches} batches...")

for i in range(0, len(vectors_to_upsert), UPSERT_BATCH_SIZE):
    batch = vectors_to_upsert[i:i + UPSERT_BATCH_SIZE]
    batch_num = i // UPSERT_BATCH_SIZE + 1
    
    try:
        index.upsert(vectors=batch)
        
        if batch_num % 20 == 0 or batch_num == total_upsert_batches:
            print(f"  Batch {batch_num}/{total_upsert_batches} uploaded")
            
    except Exception as e:
        print(f"  Error in batch {batch_num}: {e}")
        time.sleep(2)
        index.upsert(vectors=batch)

# Wait for indexing
time.sleep(2)

# Verify upload
stats = index.describe_index_stats()
print(f"\nUpload complete!")
print(f"Total vectors in index: {stats.total_vector_count}")

Uploading 4145 vectors to Pinecone in 42 batches...
  Batch 20/42 uploaded
  Batch 40/42 uploaded
  Batch 42/42 uploaded

Upload complete!
Total vectors in index: 8029


## 6. Test Vector Search

In [26]:
# Test search function
def search_products(query, top_k=5):
    """Search for products using natural language query."""
    # Get embedding for query
    query_embedding = get_embeddings_batch([query])[0]
    
    # Search in Pinecone
    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )
    
    return results.matches

# Test queries
test_queries = [
    "I need a gift for someone who loves gardening",
    "Looking for something for a child's birthday party",
    "Home decoration items",
    "Kitchen accessories",
    "Christmas decorations"
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print('='*60)
    
    results = search_products(query)
    
    for i, match in enumerate(results, 1):
        print(f"  {i}. {match.metadata['description']} (Score: {match.score:.4f}, £{match.metadata['unit_price']:.2f})")


Query: I need a gift for someone who loves gardening
  1. BOTANICAL LILY GIFT WRAP (Score: 0.4460, £0.42)
  2. BOTANICAL LILY GIFT WRAP (Score: 0.4459, £0.42)
  3. WATERING CAN GARDEN MARKER (Score: 0.4355, £1.82)
  4. WATERING CAN GARDEN MARKER (Score: 0.4354, £1.77)
  5. BOTANICAL ROSE GIFT WRAP (Score: 0.4180, £0.42)

Query: Looking for something for a child's birthday party
  1. PARTY INVITES BALLOON GIRL (Score: 0.4933, £0.95)
  2. PARTY INVITES BALLOON GIRL (Score: 0.4932, £0.96)
  3. PARTY INVITES DINOSAURS (Score: 0.4667, £0.92)
  4. PARTY INVITES DINOSAURS (Score: 0.4665, £0.91)
  5. PARTY INVITES WOODLAND (Score: 0.4563, £0.93)

Query: Home decoration items
  1. HEN HOUSE DECORATION (Score: 0.5521, £2.02)
  2. HEN HOUSE DECORATION (Score: 0.5520, £2.05)
  3. DECORATION HEN ON NEST, HANGING (Score: 0.5354, £2.50)
  4. HANGING HEART MIRROR DECORATION (Score: 0.5181, £0.89)
  5. HANGING HEART MIRROR DECORATION (Score: 0.5181, £0.90)

Query: Kitchen accessories
  1. CHILDRENS TO

## 7. Summary

In [27]:
print("="*60)
print("VECTORIZATION SUMMARY")
print("="*60)
print(f"Products embedded: {len(df_products):,}")
print(f"Embedding model: text-embedding-3-small")
print(f"Embedding dimension: 1536")
print(f"Pinecone index: {index_name}")
print(f"Vectors in index: {stats.total_vector_count:,}")
print(f"\nModule 1 (Data Preparation) Complete!")

VECTORIZATION SUMMARY
Products embedded: 4,145
Embedding model: text-embedding-3-small
Embedding dimension: 1536
Pinecone index: product-recommendations
Vectors in index: 8,029

Module 1 (Data Preparation) Complete!
